# ML Pipeline Demo

This notebook demonstrates the **full demo flow** of the project:

1. Download the last 6 months of hourly market data from Binance
2. Preprocess ETH, BTC, and ETHBTC into a single feature dataframe
3. Load the training and test configs
4. Prepare a notebook-friendly runtime that disables model persistence
5. Run the grid-search pipeline
6. Inspect selected models and example predictions

> **Note:** the demo version disables model saving, log files, and Excel outputs.  
> To keep the existing training pipeline compatible, the notebook writes the processed dataframe to a **temporary runtime CSV** only.

## 0. Optional setup for Google Colab

If you run this notebook **inside the repository**, you can skip the next cell.

If you open the notebook directly in **Colab from GitHub**, set `REPO_URL` and run the setup cell once so the rest of the project files become available in the runtime.

In [6]:
import os
import sys
import subprocess
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ixor911/ML_Pipeline_Demo.git"

if IS_COLAB and not Path("core").exists():
    if not REPO_URL:
        raise ValueError("Set REPO_URL before running this cell in Colab.")
    subprocess.run(["git", "clone", REPO_URL, "repo"], check=True)
    os.chdir("repo")
    print("Repository cloned into Colab runtime.")
else:
    print("Repository files already available. Skipping clone step.")

Repository files already available. Skipping clone step.


## 1. Download raw market data

We fetch the last 6 months of **ETHUSDT**, **BTCUSDT**, and **ETHBTC** hourly candles directly from Binance.

In [7]:
import pandas as pd
from IPython.display import display

from core.features.DataProvider import DataProvider

end = pd.Timestamp.utcnow().floor("h")
start = end - pd.DateOffset(months=6)

print(f"Downloading hourly candles from {start} to {end}...")

eth_df = DataProvider.get_ethusdt_1h(start=start, end=end)
btc_df = DataProvider.get_btcusdt_1h(start=start, end=end)
ethbtc_df = DataProvider.get_ethbtc_1h(start=start, end=end)

print(f"ETH rows    : {len(eth_df)}")
print(f"BTC rows    : {len(btc_df)}")
print(f"ETHBTC rows : {len(ethbtc_df)}")

display(eth_df.head())

ETH rows    : 4368
BTC rows    : 4368
ETHBTC rows : 4368


,open_time,open,high,low,close,volume,close_time,quote,trades,taker_buy_base,taker_buy_quote
0,2025-10-23 16:00:00+00:00,3876.95,3934.86,3865.56,3918.26,26555.1659,2025-10-23 16:59:59.999000+00:00,1.034224e+08,252272,13367.4482,5.207894e+07
1,2025-10-23 17:00:00+00:00,3918.27,3928.00,3877.04,3921.88,33951.4980,2025-10-23 17:59:59.999000+00:00,1.325764e+08,274186,15312.7095,5.979397e+07
2,2025-10-23 18:00:00+00:00,3921.88,3922.95,3858.77,3870.14,23908.6924,2025-10-23 18:59:59.999000+00:00,9.292297e+07,202490,7142.3727,2.776780e+07
3,2025-10-23 19:00:00+00:00,3870.14,3871.36,3845.00,3859.40,16742.6248,2025-10-23 19:59:59.999000+00:00,6.456791e+07,150523,7452.4666,2.874262e+07
4,2025-10-23 20:00:00+00:00,3859.40,3860.10,3818.72,3829.20,13589.4972,2025-10-23 20:59:59.999000+00:00,5.220621e+07,150297,5376.6225,2.065362e+07


## 2. Preprocess the raw candles

The preprocessing step combines ETH, BTC, and ETHBTC into one processed dataset with:
- ETH technical and candle features
- BTC context features
- ETHBTC relative-strength features
- cross-market features

In [8]:
from core.features.preprocessor.Preprocessor import Preprocessor

processed_df = Preprocessor.preprocess(
    eth_df=eth_df,
    btc_df=btc_df,
    ethbtc_df=ethbtc_df
)

print(f"Processed rows    : {len(processed_df)}")
print(f"Processed columns : {len(processed_df.columns)}")
display(processed_df.head())

Processed rows    : 4309
Processed columns : 86


,open_time_eth,open_eth,high_eth,low_eth,close_eth,volume_eth,close_time,quote_eth,trades_eth,taker_buy_base_eth,...,low_btc,open_btc,quote_btc,taker_buy_base_btc,taker_buy_quote_btc,trades_btc,volume_btc,eth_ret_1h_pct,rs_1h_pct,rs_24h_pct
0,2025-10-26 03:00:00+00:00,3930.55,3936.53,3923.55,3931.82,4500.6996,2025-10-26 03:59:59.999000+00:00,1.768499e+07,64171.0,2228.6429,...,111260.45,111448.85,1.901280e+07,69.98825,7.795710e+06,40552.0,170.69475,NaN,NaN,0.063261
1,2025-10-26 04:00:00+00:00,3931.82,3941.91,3928.83,3939.93,3983.5956,2025-10-26 04:59:59.999000+00:00,1.567642e+07,35085.0,2673.5298,...,111260.46,111260.46,1.441265e+07,62.95331,7.011840e+06,30589.0,129.38461,0.206266,0.053525,0.244469
2,2025-10-26 05:00:00+00:00,3939.94,3946.55,3937.60,3942.85,3394.1669,2025-10-26 05:59:59.999000+00:00,1.338433e+07,31331.0,1548.7680,...,111367.19,111430.39,2.341471e+07,101.82694,1.135456e+07,38827.0,209.94274,0.074113,-0.091838,0.076384
3,2025-10-26 06:00:00+00:00,3942.85,3946.41,3938.66,3944.88,3419.1602,2025-10-26 06:59:59.999000+00:00,1.348336e+07,34179.0,1559.9093,...,111548.75,111615.33,1.743419e+07,57.01429,6.363046e+06,31850.0,156.20448,0.051486,0.033002,0.168798
4,2025-10-26 07:00:00+00:00,3944.88,3956.59,3941.71,3951.57,5105.0991,2025-10-26 07:59:59.999000+00:00,2.015987e+07,36996.0,2863.6659,...,111620.02,111635.96,1.898354e+07,73.31342,8.187782e+06,33958.0,169.97811,0.169587,0.090732,0.366268


## 3. Load training and test configs

Here we load:

- `model_train_basic_grid` — the training grid
- `model_test_basic` — the test slicing config

In [9]:
from pprint import pprint

from core.io.loader import load_config, load_config_grid

training_configs = list(load_config_grid("model_train_basic_grid"))
test_config = load_config("model_test_basic")

print(f"Training configs loaded: {len(training_configs)}")
print("\nExample training config:")
pprint(training_configs[0])

print("\nTest config:")
pprint(test_config)

Training configs loaded: 40

Example training config:
{'builder': {'extra_drop': None,
             'feature_groups': ['eth_trend_core', 'eth_macd_pack'],
             'features_include': None,
             'future_ret_col': 'future_ret',
             'horizon': 1,
             'regime_filter': ['trend_up'],
             'tau_pct': [0.03, None]},
 'datapath': 'ETHUSDT_6_1h.csv',
 'model': {'batch_size': 64,
           'depth': 3,
           'dropout': 0.2,
           'epochs': 25,
           'hidden': 64,
           'lr': 0.001,
           'patience': 6,
           'seed': 42,
           'task': 'classification',
           'val_share': 0.2,
           'verbose': False,
           'weight_decay': 0.0001},
 'slicing': {'candles': [3000, 1000],
             'end_date': None,
             'from_tail': True,
             'time_col': 'open_time_eth',
             'type': 'candles'}}

Test config:
{'datapath': 'ETHUSDT_6_1h.csv',
 'slicing': {'candles': [1000],
             'end_date': None,

## 4. Build test candles from the processed dataframe

The grid-search pipeline evaluates all candidates on one shared test candle window.
We build that window from the freshly processed dataset using the test slicing config.


In [11]:
from core.target.slicing import slice_data

test_windows = slice_data(
    df=processed_df,
    slicing_config=test_config["slicing"],
)
test_candles = test_windows[0]

print(f"Test candles rows: {len(test_candles)}")
display(test_candles.head())


Test candles rows: 1000


,open_time_eth,open_eth,high_eth,low_eth,close_eth,volume_eth,close_time,quote_eth,trades_eth,taker_buy_base_eth,...,low_btc,open_btc,quote_btc,taker_buy_base_btc,taker_buy_quote_btc,trades_btc,volume_btc,eth_ret_1h_pct,rs_1h_pct,rs_24h_pct
0,2026-03-13 00:00:00+00:00,2073.51,2148.80,2070.12,2119.19,85178.5833,2026-03-13 00:59:59.999000+00:00,1.807812e+08,634499.0,40597.0062,...,70386.01,70541.34,2.040037e+08,1521.98144,1.086833e+08,447875.0,2857.31114,2.202535,0.878106,1.304105
1,2026-03-13 01:00:00+00:00,2119.19,2130.40,2111.72,2124.76,15741.6540,2026-03-13 01:59:59.999000+00:00,3.341860e+07,258154.0,8055.7323,...,71283.99,71475.60,6.983872e+07,508.42321,3.639448e+07,207345.0,975.74666,0.262836,-0.012474,1.215510
2,2026-03-13 02:00:00+00:00,2124.75,2127.12,2102.42,2105.81,13893.8920,2026-03-13 02:59:59.999000+00:00,2.937212e+07,186281.0,5631.9422,...,71045.52,71672.38,7.105742e+07,444.89169,3.177939e+07,147297.0,994.81538,-0.891865,-0.154357,1.327047
3,2026-03-13 03:00:00+00:00,2105.81,2123.58,2102.20,2118.85,11965.2095,2026-03-13 03:59:59.999000+00:00,2.525016e+07,133309.0,5766.3768,...,70964.43,71143.80,6.605695e+07,517.00627,3.681469e+07,125941.0,927.82134,0.619239,0.251041,1.897269
4,2026-03-13 04:00:00+00:00,2118.86,2120.06,2109.63,2111.68,5461.6899,2026-03-13 04:59:59.999000+00:00,1.154701e+07,89249.0,2411.5234,...,71214.01,71405.76,5.570447e+07,429.05759,3.061340e+07,105161.0,780.92498,-0.338391,-0.321852,1.464311


## 5. Run the async grid-search pipeline

This step:
- trains models for all training configs
- expands each trained model into category-specific candidates
- evaluates them on the shared test candles
- keeps only the strongest candidates in `models_ram`

> In Colab, start with small values for `WORKERS` and `MAX_CONFIGS` if you want a faster first run.


In [15]:
import os

from core.pipelines.grid_search_pipeline_async import (
    grid_search_pipeline_async,
    print_models_ram_metrics,
)

LIMIT = 3
WORKERS = 1
QUEUE_MAXSIZE = 32

models_ram = {}

models_ram = grid_search_pipeline_async(
    training_configs=training_configs,
    test_candles=test_candles,
    models_ram=models_ram,
    limit=LIMIT,
    workers=WORKERS,
    queue_maxsize=QUEUE_MAXSIZE,
)

print("Grid search finished.")


[1 / 40] config finished | worker=1 | tested_states=5 | remaining=39
[2 / 40] config finished | worker=1 | tested_states=5 | remaining=38
[3 / 40] config finished | worker=1 | tested_states=5 | remaining=37
[4 / 40] config finished | worker=1 | tested_states=5 | remaining=36
[5 / 40] config finished | worker=1 | tested_states=5 | remaining=35
[6 / 40] config finished | worker=1 | tested_states=5 | remaining=34
[7 / 40] config finished | worker=1 | tested_states=5 | remaining=33
[8 / 40] config finished | worker=1 | tested_states=5 | remaining=32
[9 / 40] config finished | worker=1 | tested_states=5 | remaining=31
[10 / 40] config finished | worker=1 | tested_states=5 | remaining=30
[11 / 40] config finished | worker=1 | tested_states=5 | remaining=29
[12 / 40] config finished | worker=1 | tested_states=5 | remaining=28
[13 / 40] config finished | worker=1 | tested_states=5 | remaining=27
[14 / 40] config finished | worker=1 | tested_states=5 | remaining=26
[15 / 40] config finished | w

FileNotFoundError: [Errno 2] No such file or directory

## 7. Inspect the final in-memory model pool

In [ ]:
def count_models(models_ram: dict) -> int:
    return sum(len(category_models) for category_models in models_ram.values())

def count_categories(models_ram: dict) -> int:
    return len(models_ram)

print(f"Categories kept : {count_categories(models_ram)}")
print(f"Models kept     : {count_models(models_ram)}")

print_models_ram_metrics(models_ram=models_ram)


## 9. What this demo notebook shows

This notebook demonstrates the main ML workflow of the demo project:

- downloading recent market data
- preprocessing ETH / BTC / ETHBTC candles
- loading train/test configs
- running async grid search
- keeping the strongest candidate models in memory
- inspecting final predictions

The full private version of the project also includes:
- model persistence and loading
- model metadata analysis
- backtesting and grid-backtesting
- system-management helper pipelines
- ongoing work on step-by-step retraining and AI-agent integration
